# Part E: remaining reviewer-proofing experiments

Run this **after** `xai_comparison_harness_v5.ipynb` (with `PREPROCESSING = 'log2_loggenes'`) and Part D. It reads the v5
checkpoints in `pancreatic_paper_checkpoints_log2_loggenes/` (and, if present, Part D's expression file) and writes
everything new to `pancreatic_paper_checkpoints_log2_loggenes/partE/`. Nothing from v5 or Part D is overwritten.
Every slow step checkpoints, so after a disconnect just **Run all** again and finished steps are skipped.

| Cell | What it does | Reviewer point it answers | CPU time (approx.) |
|---|---|---|---|
| E0-E1 | setup; loads the v5 data, final model, fold models and saved explanations; reproduction checks | - | 3 min |
| E2 | three new explainers on the full cohort: converged KernelSHAP (4,096 coalitions), GradientSHAP, DeepLIFT | "Table 5 compares IG with a SHAP you showed is broken"; "why KernelSHAP for a PyTorch net?" | 15 min |
| E3 | in-sample faithfulness under three fill-ins (median = v5, mean, resampled patients) with bootstrap CIs and tests vs IG | "every metric shares IG's median baseline" | 10 min |
| E4 | **out-of-fold** faithfulness and fold stability (fold models explain their own held-out patients) | "the full-cohort model has in-sample C = 0.998" | 30 min |
| E5 | ROAR-style retraining after removing each method's top features | removal without distribution shift | 12 min |
| E6 | sanity check, robustness, linear ground truth, complexity for the new methods | complete Table 5 rows | 35 min |
| E7 | planted ground truth with 10 + 3 replicates and two signal strengths, all 8 methods | "3 replicates is too few" | 90 min |
| E8 | Moffitt basal-like / classical subtypes vs predictions and attributions | biological validation | 2 min |
| E9 | summary: Table 5 (two parts) with CIs and significance markers, rank table, JSON, LaTeX | - | seconds |

**First run with `QUICK_RUN = True`** (about 15 minutes, tiny budgets, writes to `partE_quick/`; numbers are NOT for the
paper). If it finishes without errors, set `QUICK_RUN = False` in E0 and Run all again.

In [ ]:
# ===== E0. Setup =====
import os, sys, subprocess, json, pickle, time, copy, warnings, urllib.request
from itertools import combinations

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lifelines', 'shap', 'lime', 'captum',
                    'scikit-survival'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')

QUICK_RUN = False     # True = ~15-minute dry run with tiny budgets (numbers NOT for the paper). Set False for the real run.
BASE      = os.environ.get('PAPER_BASE', '/content/drive/MyDrive')
DATA_DIR  = f'{BASE}/pancreatic_paper_data'
V5_DIR    = f'{BASE}/pancreatic_paper_checkpoints_log2_loggenes'     # the verified v5 run
PARTD_DIR = f'{V5_DIR}/partD'
OUT       = f'{V5_DIR}/partE' + ('_quick' if QUICK_RUN else '')
os.makedirs(OUT, exist_ok=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, KFold, GridSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import average_precision_score
from sklearn.cluster import KMeans
from scipy.stats import spearmanr, rankdata, wilcoxon, friedmanchisquare, hypergeom
from lifelines import CoxPHFitter
from lifelines.statistics import logrank_test
from lime.lime_tabular import LimeTabularExplainer
from captum.attr import IntegratedGradients, GradientShap, DeepLift
warnings.filterwarnings('ignore')
torch.set_num_threads(max(1, os.cpu_count() or 1))

# ---- budgets (the real run uses the paper's settings) ----
SHAP_LO      = 200                              # the v5 configuration (with l1_reg = num_features(50))
SHAP_HI      = 256 if QUICK_RUN else 4096       # converged KernelSHAP (v5 F1: rho = 0.96 with IG)
GS_SAMPLES   = 10 if QUICK_RUN else 50          # GradientSHAP samples per patient
N_BOOT       = 100 if QUICK_RUN else 1000       # patient bootstrap resamples
N_RANDOM     = 5 if QUICK_RUN else 20           # random feature sets for deletion / insertion baselines
EPOCHS       = 30 if QUICK_RUN else 300
PFI_REPEATS  = 2 if QUICK_RUN else 10
SYN_SETTINGS = ([(1.5, [1000])] if QUICK_RUN else
                [(1.5, list(range(1000, 1010))), (3.0, list(range(2000, 2003)))])   # (signal strength, seeds)
SYN_SHAP_HI_N = 10 if QUICK_RUN else 60         # patients explained by SHAP-4096 in each planted replicate
ROAR_K       = [10, 50] if QUICK_RUN else [10, 50, 100]
ROAR_RANDOM  = 2 if QUICK_RUN else 5


def save(name, obj):
    path = f'{OUT}/{name}.pkl'
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'wb') as f:
        pickle.dump(obj, f)


def load(name):
    path = f'{OUT}/{name}.pkl'
    if os.path.exists(path):
        with open(path, 'rb') as f:
            return pickle.load(f)
    return None


def load_v5(name):
    path = f'{V5_DIR}/{name}.pkl'
    if os.path.exists(path):
        with open(path, 'rb') as f:
            return pickle.load(f)
    return None


RESULTS = load('results') or {}   # every headline number lands here; E9 writes it to JSON


def record(key, value):
    RESULTS[key] = value
    save('results', RESULTS)


METHODS  = ['SHAP-200', 'SHAP-4096', 'LIME', 'IG', 'GradSHAP', 'DeepLIFT', 'PFI', 'PDP']
LOCAL_M  = ['SHAP-200', 'SHAP-4096', 'LIME', 'IG', 'GradSHAP', 'DeepLIFT']
GLOBAL_M = ['PFI', 'PDP']
print(f"QUICK_RUN = {QUICK_RUN}; outputs -> {OUT}")

In [ ]:
# ===== E1. Data, models, helpers, and the saved v5 results (nothing here retrains the paper's models) =====
DATA_CSV = f'{DATA_DIR}/tcga_paad_clean_log2.csv'
df = pd.read_csv(DATA_CSV)
stage_map = {'STAGE I': 1, 'STAGE IA': 1, 'STAGE IB': 1, 'STAGE II': 2, 'STAGE IIA': 2, 'STAGE IIB': 2,
             'STAGE III': 3, 'STAGE IV': 4}
df['STAGE_ENCODED'] = df['AJCC_PATHOLOGIC_TUMOR_STAGE'].map(stage_map).fillna(2)
gene_cols = df.columns[5:1005].tolist()
CLIN_COLS = ['AGE', 'STAGE_ENCODED']
NC = len(CLIN_COLS)
ID_COL = next(c for c in df.columns[:5] if df[c].astype(str).str.startswith('TCGA-').all())
PID = df[ID_COL].astype(str).str[:12].values
N = len(df)
f32 = np.float32

Xc_raw = df[CLIN_COLS].values.astype(f32)
Xg_log = df[gene_cols].values.astype(f32)
if Xg_log.max() > 100:
    Xg_log = np.log2(Xg_log + 1)


def clip_pct(A, ref=None):
    ref = A if ref is None else ref
    return np.clip(A, np.percentile(ref, 1, axis=0), np.percentile(ref, 99, axis=0)).astype(f32)


Xg_clip = clip_pct(Xg_log)
T = df['OS_MONTHS'].values.astype(f32)
E = df['OS_STATUS'].values.astype(f32)
E_int = E.astype(int)
FEATURES = CLIN_COLS + gene_cols
FOLDS = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(Xc_raw, E))
print(f"{N} patients, {len(gene_cols)} genes, {int(E.sum())} deaths")


def cox_ph_loss(risk, T_, E_):
    order = torch.argsort(T_, descending=True)
    risk, E_ = risk[order], E_[order]
    return -torch.mean((risk - torch.logcumsumexp(risk, dim=0)) * E_)


class GatedFusion(nn.Module):
    kind = 'fusion'

    def __init__(self, n_clin=2, n_gen=1000):
        super().__init__()
        self.clin_encoder = nn.Sequential(nn.Linear(n_clin, 32), nn.BatchNorm1d(32), nn.ReLU(),
                                          nn.Linear(32, 64), nn.BatchNorm1d(64), nn.ReLU())
        self.gen_encoder = nn.Sequential(nn.Linear(n_gen, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4),
                                         nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.4))
        self.gate_clin = nn.Sequential(nn.Linear(192, 64), nn.Sigmoid())
        self.gate_gen = nn.Sequential(nn.Linear(192, 128), nn.Sigmoid())
        self.head = nn.Linear(192, 1)

    def forward(self, xc, xg):
        hc, hg = self.clin_encoder(xc), self.gen_encoder(xg)
        h = torch.cat([hc, hg], 1)
        return self.head(torch.cat([hc * self.gate_clin(h), hg * self.gate_gen(h)], 1)).squeeze(1)


GatedFusionModel = GatedFusion     # the v5 checkpoints were saved under this class name


def _forward(model, xc, xg):
    return model(xc, xg)


def train_model(model, Xc, Xg, T_, E_, epochs=None, lr=1e-3):
    epochs = epochs or EPOCHS
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-3)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    Xc, Xg = torch.tensor(Xc, dtype=torch.float32), torch.tensor(Xg, dtype=torch.float32)
    Tt, Et = torch.tensor(T_, dtype=torch.float32), torch.tensor(E_, dtype=torch.float32)
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        loss = cox_ph_loss(model(Xc, Xg), Tt, Et)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sch.step()
    return model.eval()


def risk_of(model, Xc, Xg):
    with torch.no_grad():
        return model(torch.tensor(Xc, dtype=torch.float32), torch.tensor(Xg, dtype=torch.float32)).numpy()


class Flat(nn.Module):
    """f([clinical | genes]) for the explainers."""
    def __init__(self, base, nc=NC):
        super().__init__()
        self.base, self.nc = base, nc

    def forward(self, x):
        return self.base(x[:, :self.nc], x[:, self.nc:])


class LinearFlat(nn.Module):
    """A linear Cox model f(x) = beta . x, for the analytic ground truth."""
    def __init__(self, beta):
        super().__init__()
        self.lin = nn.Linear(len(beta), 1, bias=False)
        with torch.no_grad():
            self.lin.weight.copy_(torch.tensor(np.asarray(beta, dtype=f32))[None])

    def forward(self, x):
        return self.lin(x).squeeze(1)


def np_fn(module):
    def fn(X):
        with torch.no_grad():
            return module(torch.tensor(np.asarray(X, dtype=f32))).numpy()
    return fn


def harrell_c_many(T_, E_, R):
    """Harrell's C for every row of R; same pairing and tie rules as lifelines.concordance_index(T, -risk, E)."""
    T_ = np.asarray(T_, float); E_ = np.asarray(E_).astype(bool); R = np.atleast_2d(np.asarray(R, float))
    comp = ((T_[:, None] < T_[None, :]) | ((T_[:, None] == T_[None, :]) & ~E_[None, :])) & E_[:, None]
    n = comp.sum()
    if n == 0:
        return np.full(len(R), np.nan)
    out = np.empty(len(R))
    for s in range(0, len(R), 64):                     # chunks keep memory small
        Rc = R[s:s + 64]
        gt = (Rc[:, :, None] > Rc[:, None, :]) & comp
        eq = (Rc[:, :, None] == Rc[:, None, :]) & comp
        out[s:s + 64] = (gt.sum((1, 2)) + 0.5 * eq.sum((1, 2))) / n
    return out


def harrell_c(T_, E_, r):
    return float(harrell_c_many(T_, E_, np.asarray(r)[None])[0])


# ---- explainers (identical settings to v5 for SHAP-200, LIME, IG) ----
def ig_attr(module, X, baseline_vec, n_steps=50):
    X_t = torch.tensor(np.asarray(X, dtype=f32), requires_grad=True)
    base = torch.tensor(np.tile(np.asarray(baseline_vec, dtype=f32), (len(X), 1)))
    return IntegratedGradients(module).attribute(X_t, baselines=base, n_steps=n_steps,
                                                 internal_batch_size=4096).detach().numpy()


def kernel_shap(fn, X, bg_pool, nsamples, seed=42, l1='num_features(50)', n_bg=30):
    """v5 KernelSHAP: background = shap.sample(pool, 30) (random_state 0); `seed` sets the coalition sampling."""
    np.random.seed(seed)
    ex = shap.KernelExplainer(fn, shap.sample(bg_pool, min(n_bg, len(bg_pool))))
    sv = ex.shap_values(np.asarray(X, dtype=f32), nsamples=nsamples, l1_reg=l1, silent=True)
    return np.asarray(sv).reshape(len(X), -1)


def lime_local(fn, X_explain, X_train, num_samples=300, seed=42):
    ex = LimeTabularExplainer(training_data=X_train, mode='regression', discretize_continuous=False, random_state=seed)
    out = np.zeros(X_explain.shape, dtype=float)
    for i in range(len(X_explain)):
        e = ex.explain_instance(X_explain[i], fn, num_features=X_explain.shape[1], num_samples=num_samples)
        for j, w in e.as_map()[1]:
            out[i, j] = w
    return out


def gradshap_attr(module, X, background, n_samples=None, seed=42):
    """GradientSHAP (expected gradients) with the same 30-patient background as KernelSHAP."""
    n_samples = n_samples or GS_SAMPLES
    torch.manual_seed(seed)
    B = torch.tensor(np.asarray(background, dtype=f32))
    out = []
    for s in range(0, len(X), 32):
        Xb = torch.tensor(np.asarray(X[s:s + 32], dtype=f32), requires_grad=True)
        out.append(GradientShap(module).attribute(Xb, baselines=B, n_samples=n_samples, stdevs=0.0).detach().numpy())
    return np.concatenate(out)


def deeplift_attr(module, X, baseline_vec):
    """DeepLIFT (Rescale rule) from the same median baseline as IG. Returns attributions and completeness gaps."""
    X_t = torch.tensor(np.asarray(X, dtype=f32), requires_grad=True)
    base = torch.tensor(np.tile(np.asarray(baseline_vec, dtype=f32), (len(X), 1)))
    a, d = DeepLift(module).attribute(X_t, baselines=base, return_convergence_delta=True)
    return a.detach().numpy(), d.detach().numpy()


def pfi_global(fn, X, T_, E_, n_repeats=None, seed=42):
    """Permutation importance: mean C-index drop when a feature is permuted (vectorised over repeats)."""
    n_repeats = n_repeats or PFI_REPEATS
    rng = np.random.RandomState(seed)
    c0 = harrell_c(T_, E_, fn(X))
    imp = np.zeros(X.shape[1])
    for j in range(X.shape[1]):
        Xs = np.repeat(X[None], n_repeats, axis=0)
        for r in range(n_repeats):
            Xs[r, :, j] = X[rng.permutation(len(X)), j]
        preds = fn(Xs.reshape(-1, X.shape[1])).reshape(n_repeats, len(X))
        imp[j] = c0 - harrell_c_many(T_, E_, preds).mean()
    return imp


def pdp_global(fn, X, cand, n_grid=10):
    out = np.zeros(X.shape[1])
    for j in cand:
        grid = np.percentile(X[:, j], np.linspace(10, 90, n_grid))
        Xs = np.repeat(X[None], n_grid, axis=0)
        Xs[np.arange(n_grid), :, j] = grid[:, None]
        curve = fn(Xs.reshape(-1, X.shape[1])).reshape(n_grid, len(X)).mean(1)
        out[j] = curve.max() - curve.min()
    return out


def randomise(model, seed=0):
    torch.manual_seed(seed)
    rm = copy.deepcopy(model)
    for mod in rm.modules():
        if isinstance(mod, nn.Linear):
            mod.reset_parameters()
    return rm.eval()


def top_genes(g, names=FEATURES, k=20, nc=NC):
    o = np.argsort(-np.asarray(g))
    return [names[i] for i in o if i >= nc][:k]


def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / max(1, len(a | b))


def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for r, i in enumerate(order):
        run = max(run, min(1.0, (len(p) - r) * p[i])); adj[i] = run
    return adj


def vs_ig_wilcoxon(W, higher_better):
    """W: DataFrame (rows = patients, cols = methods). Holm-corrected Wilcoxon of every method vs IG."""
    W = W.dropna(axis=1, how='all')
    others = [m for m in W.columns if m != 'IG']
    p = [wilcoxon(W[m], W['IG']).pvalue if np.any(W[m] != W['IG']) else 1.0 for m in others]
    adj = holm(p) if p else []
    return {m: float(a) for m, a in zip(others, adj)}


def vs_ig_boot(boot, higher_better):
    """boot: dict method -> bootstrap replicates (paired). Two-sided p of (method - IG), Holm-corrected."""
    others = [m for m in boot if m != 'IG']
    p = []
    for m in others:
        d = boot[m] - boot['IG']; d = d[np.isfinite(d)]
        p.append(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())) if len(d) else np.nan)
    adj = holm(p) if p else []
    return {m: float(a) for m, a in zip(others, adj)}


# ---- saved v5 results ----
def need(name):
    obj = load_v5(name)
    if obj is None:
        raise FileNotFoundError(f"{name}.pkl not found in {V5_DIR}. Run the v5 notebook with PREPROCESSING = "
                                f"'log2_loggenes' first.")
    return obj


cv_state = need('cv_state')
FOLD_MODELS = [m.eval() for m in cv_state['fold_fusion_models']]
FOLD_SCALERS = cv_state['fold_scalers']
assert all(np.array_equal(np.sort(te), np.sort(t5)) for (tr, te), t5 in zip(FOLDS, cv_state['fold_test_indices'])), \
    "Fold split differs from v5"
_ms = need('model_full_state')
model_full = GatedFusion(); model_full.load_state_dict(_ms['model_full_state_dict']); model_full.eval()
X_all = _ms['X_all'].astype(f32)
F_full = Flat(model_full).eval()
pred_full = np_fn(F_full)
r_all = pred_full(X_all)
MED_ALL = np.median(X_all, axis=0).astype(f32)
c_orig = harrell_c(T, E_int, r_all)
_names = load_v5('extended_eval_feature_names')
assert _names is None or list(_names) == FEATURES, "Feature order differs from v5"

_g5 = need('extended_eval_globals')
GI5 = _g5['global_importances']
CAND = sorted(int(c) for c in _g5['candidates'])
RUNTIME = {'SHAP-200': _g5['runtimes']['SHAP'], 'LIME': _g5['runtimes']['LIME'], 'IG': _g5['runtimes']['IG'],
           'PFI': _g5['runtimes']['PFI'], 'PDP': _g5['runtimes']['PDP']}
_loc5 = need('extended_eval_local_full')
LOCAL = {'SHAP-200': np.asarray(need('shap_state')['shap_values']).reshape(N, -1),
         'LIME': _loc5['LIME'], 'IG': _loc5['IG']}
GI = {'SHAP-200': GI5['SHAP'], 'LIME': GI5['LIME'], 'IG': GI5['IG'], 'PFI': GI5['PFI'], 'PDP': GI5['PDP']}
SAN5 = need('extended_eval_sanity')
ROB5 = need('extended_eval_robustness')
LIN5 = need('addon_linear_gt')
OOF5 = need('extended_eval_oof_preds')
CONV5 = load_v5('addon_shap_convergence')
BG30 = shap.sample(X_all, 30)          # the KernelSHAP background used in v5 (random_state 0)

print(f"Final model in-sample C = {c_orig:.4f} (v5: 0.9976); IG top-5 genes: {top_genes(GI['IG'], k=5)}")
_chk = [harrell_c(T[te], E_int[te], risk_of(m, *(lambda s: (s[0].transform(Xc_raw[te]).astype(f32),
                                                             s[1].transform(Xg_clip[te]).astype(f32)))(sc)))
        for (tr, te), m, sc in zip(FOLDS, FOLD_MODELS, FOLD_SCALERS)]
print(f"v5 fold models, held-out C: {np.round(_chk, 4)} (mean {np.mean(_chk):.4f}; v5 gated fusion 0.5958)")

In [ ]:
# ===== E2. Three new explainers on the full cohort =====
# SHAP-4096: KernelSHAP without L1 truncation at the converged budget (same seed and background as v5 cell F1).
# GradientSHAP: expected gradients over the same 30-patient background - the standard SHAP variant for PyTorch.
# DeepLIFT (Rescale): from the same median baseline as IG.
ck = load('E2_shap_hi')
if ck is None:
    t0 = time.time()
    ck = {'local': kernel_shap(pred_full, X_all, X_all, SHAP_HI, seed=42, l1=False), 'runtime': None}
    ck['runtime'] = time.time() - t0
    save('E2_shap_hi', ck)
LOCAL['SHAP-4096'], RUNTIME['SHAP-4096'] = ck['local'], ck['runtime']

ck = load('E2_grad')
if ck is None:
    t0 = time.time(); gs = gradshap_attr(F_full, X_all, BG30, seed=42); t_gs = time.time() - t0
    t0 = time.time(); dl, dl_delta = deeplift_attr(F_full, X_all, MED_ALL); t_dl = time.time() - t0
    ck = {'GradSHAP': gs, 'DeepLIFT': dl, 'dl_delta': dl_delta, 't_gs': t_gs, 't_dl': t_dl}
    save('E2_grad', ck)
LOCAL['GradSHAP'], LOCAL['DeepLIFT'] = ck['GradSHAP'], ck['DeepLIFT']
RUNTIME['GradSHAP'], RUNTIME['DeepLIFT'] = ck['t_gs'], ck['t_dl']
for m in ['SHAP-4096', 'GradSHAP', 'DeepLIFT']:
    GI[m] = np.abs(LOCAL[m]).mean(0)

# Checks
gap_dl = np.abs(ck['dl_delta']) / (np.abs(r_all - pred_full(MED_ALL[None]))[0] + 1e-8)
print(f"DeepLIFT completeness gap: median {np.median(np.abs(ck['dl_delta'])):.2e} "
      f"(relative to |f(x) - f(baseline)|: median {np.median(gap_dl):.3f}). DeepLIFT's Rescale rule is exact only for "
      f"piecewise-linear nets; the multiplicative gates make it approximate.")
if CONV5 is not None and max(CONV5) == SHAP_HI:
    print(f"SHAP-{SHAP_HI} reproduces v5 cell F1: rho = {spearmanr(GI['SHAP-4096'], CONV5[SHAP_HI]['global']).correlation:.4f}")
rows = []
for m in METHODS:
    mask = np.isin(np.arange(len(FEATURES)), CAND) if m == 'PDP' else np.ones(len(FEATURES), bool)
    rows.append({'method': m, 'rho_vs_IG': spearmanr(GI[m][mask], GI['IG'][mask]).correlation,
                 'rho_vs_SHAP4096': spearmanr(GI[m][mask], GI['SHAP-4096'][mask]).correlation,
                 'top20_genes_shared_with_IG': len(set(top_genes(GI[m])) & set(top_genes(GI['IG']))),
                 'runtime_s': RUNTIME[m], 'top5_genes': ', '.join(top_genes(GI[m], k=5))})
E2 = pd.DataFrame(rows)
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 70)
print(E2.round(3).to_string(index=False))
E2.to_csv(f'{OUT}/E2_new_explainers.csv', index=False)
record('E2_new_explainers', E2.to_dict('records'))

In [ ]:
# ===== E3. In-sample faithfulness under three fill-ins, with bootstrap CIs and tests against IG =====
# 'median'   : features set to the cohort median (= v5; also IG's baseline, so it may favour IG)
# 'mean'     : features set to the cohort mean
# 'resample' : features replaced by the values of randomly drawn other patients (interventional removal;
#              predictions averaged over draws) - no single reference point, so no method's baseline is favoured
KS = [1, 2, 5, 10, 20, 50, 100]
KS_LOCAL = [5, 10, 20]
FILLS = ['median', 'mean', 'resample']
N_DRAWS = 2 if QUICK_RUN else 5


def filled_pred(fn, X, cols, mode, ref, rng):
    """Model output when columns `cols` of X are removed with the given fill-in (ref = reference patients)."""
    draws = N_DRAWS if mode == 'resample' else 1
    out = 0.0
    for _ in range(draws):
        Xp = X.copy()
        if len(cols):
            if mode == 'median':
                Xp[:, cols] = np.median(ref, 0)[cols]
            elif mode == 'mean':
                Xp[:, cols] = ref.mean(0)[cols]
            else:
                Xp[:, cols] = ref[rng.randint(0, len(ref), len(X))][:, cols]
        out = out + fn(Xp)
    return out / draws


def make_stack(fn, X, orders, mode, ref, seed=42):
    """Predictions for deletion / insertion of each method's top-k and of random k-sets; last row = unmodified."""
    rs = np.random.RandomState(seed); rng = np.random.RandomState(seed + 1)
    p = X.shape[1]; allc = np.arange(p)
    rnd = [[rs.choice(p, k, replace=False) for k in KS] for _ in range(N_RANDOM)]
    rows = []
    for m in orders:
        rows += [filled_pred(fn, X, orders[m][:k], mode, ref, rng) for k in KS]
    for m in orders:
        rows += [filled_pred(fn, X, np.setdiff1d(allc, orders[m][:k]), mode, ref, rng) for k in KS]
    for r in rnd:
        rows += [filled_pred(fn, X, ix, mode, ref, rng) for ix in r]
    for r in rnd:
        rows += [filled_pred(fn, X, np.setdiff1d(allc, ix), mode, ref, rng) for ix in r]
    rows.append(fn(X))
    return np.stack(rows)


def stack_stats(T_, E_, S, n_m, ii):
    K_ = len(KS)
    C = harrell_c_many(T_[ii], E_[ii], S[:, ii])
    a, b, c = n_m * K_, 2 * n_m * K_, 2 * n_m * K_ + N_RANDOM * K_
    cd, ci = C[:a].reshape(n_m, K_), C[a:b].reshape(n_m, K_)
    rd, ri = C[b:c].reshape(N_RANDOM, K_), C[c:-1].reshape(N_RANDOM, K_)
    c0 = C[-1]
    drop = c0 - cd
    return {'drop': drop, 'ins': ci, 'del_auc': drop.mean(1) - (c0 - rd).mean(), 'ins_auc': ci.mean(1) - ri.mean(),
            'rnd_drop': (c0 - rd).mean(0), 'rnd_ins': ri.mean(0), 'c0': c0}


def evaluate_stacks(parts, n_m, seed=7):
    """parts: list of (T, E, STACK) - one per fold (or one for the whole cohort). Point estimate = mean over parts;
    CIs from resampling patients within each part."""
    def stats(idx_list):
        ss = [stack_stats(T_, E_, S, n_m, ii) for (T_, E_, S), ii in zip(parts, idx_list)]
        return {k: np.nanmean([s[k] for s in ss], axis=0) for k in ss[0]}
    point = stats([np.arange(len(T_)) for T_, _, _ in parts])
    rb = np.random.RandomState(seed)
    boots = [stats([rb.randint(0, len(T_), len(T_)) for T_, _, _ in parts]) for _ in range(N_BOOT)]
    return point, {k: np.array([b[k] for b in boots]) for k in ('del_auc', 'ins_auc')}


def faith_table(point, boot, label):
    rows = []
    p_del = vs_ig_boot({m: boot['del_auc'][:, i] for i, m in enumerate(METHODS)}, True)
    p_ins = vs_ig_boot({m: boot['ins_auc'][:, i] for i, m in enumerate(METHODS)}, True)
    for i, m in enumerate(METHODS):
        rows.append({'method': m, 'fill': label, 'faith@10': point['drop'][i, KS.index(10)],
                     'del_auc': point['del_auc'][i], 'del_lo': np.nanpercentile(boot['del_auc'][:, i], 2.5),
                     'del_hi': np.nanpercentile(boot['del_auc'][:, i], 97.5), 'del_p_vs_IG': p_del.get(m, np.nan),
                     'ins_auc': point['ins_auc'][i], 'ins_lo': np.nanpercentile(boot['ins_auc'][:, i], 2.5),
                     'ins_hi': np.nanpercentile(boot['ins_auc'][:, i], 97.5), 'ins_p_vs_IG': p_ins.get(m, np.nan)})
    return pd.DataFrame(rows)


def comp_suff(fn, X, attr_by_method, mode, ref, sd):
    """Per-patient comprehensiveness / sufficiency (risk change in units of sd), mean over k = 5, 10, 20."""
    rng = np.random.RandomState(3)
    r0 = fn(X)
    comp, suff = {}, {}
    for m, A in attr_by_method.items():
        imp = np.abs(A) if A.ndim == 2 else np.tile(A, (len(X), 1))
        c_acc, s_acc = np.zeros(len(X)), np.zeros(len(X))
        for k in KS_LOCAL:
            top = np.argsort(-imp, axis=1)[:, :k]
            draws = N_DRAWS if mode == 'resample' else 1
            pd_, ps_ = 0.0, 0.0
            for _ in range(draws):
                if mode == 'median':
                    fillrow = np.tile(np.median(ref, 0), (len(X), 1))
                elif mode == 'mean':
                    fillrow = np.tile(ref.mean(0), (len(X), 1))
                else:
                    fillrow = ref[rng.randint(0, len(ref), len(X))].copy()
                Xd = X.copy(); np.put_along_axis(Xd, top, np.take_along_axis(fillrow, top, 1), axis=1)
                Xk = fillrow.copy(); np.put_along_axis(Xk, top, np.take_along_axis(X, top, 1), axis=1)
                pd_ = pd_ + fn(Xd); ps_ = ps_ + fn(Xk)
            c_acc += np.abs(r0 - pd_ / draws) / sd
            s_acc += np.abs(r0 - ps_ / draws) / sd
        comp[m], suff[m] = c_acc / len(KS_LOCAL), s_acc / len(KS_LOCAL)
    return pd.DataFrame(comp), pd.DataFrame(suff)


ORDERS_ALL = {m: np.argsort(-GI[m]) for m in METHODS}
ATTR_ALL = {m: (LOCAL[m] if m in LOCAL_M else GI[m]) for m in METHODS}
E3_tabs, E3_cs = [], []
for mode in FILLS:
    ck = load(f'E3_{mode}')
    if ck is None:
        S = make_stack(pred_full, X_all, ORDERS_ALL, mode, X_all)
        point, boot = evaluate_stacks([(T, E_int, S)], len(METHODS))
        cp, sf = comp_suff(pred_full, X_all, ATTR_ALL, mode, X_all, r_all.std())
        ck = {'point': point, 'boot': boot, 'comp': cp, 'suff': sf}
        save(f'E3_{mode}', ck)
    E3_tabs.append(faith_table(ck['point'], ck['boot'], mode))
    p_c, p_s = vs_ig_wilcoxon(ck['comp'], True), vs_ig_wilcoxon(ck['suff'], False)
    E3_cs.append(pd.DataFrame({'method': METHODS, 'fill': mode, 'comp': ck['comp'].mean().values,
                               'comp_p_vs_IG': [p_c.get(m, np.nan) for m in METHODS],
                               'suff': ck['suff'].mean().values, 'suff_p_vs_IG': [p_s.get(m, np.nan) for m in METHODS]}))
E3 = pd.concat(E3_tabs, ignore_index=True).merge(pd.concat(E3_cs, ignore_index=True), on=['method', 'fill'])
print("In-sample faithfulness (full-cohort model). del/ins = AUC in excess of random features; p = Holm vs IG.")
print(E3.round(4).to_string(index=False))
_v5m1 = f'{V5_DIR}/extended_eval/M1_deletion_insertion.csv'
if os.path.exists(_v5m1) and not QUICK_RUN:
    v5m1 = pd.read_csv(_v5m1).set_index('method')
    med = E3[E3.fill == 'median'].set_index('method')
    print("\nReproduction of v5 (median fill) deletion AUC:",
          {m5: (round(med.loc[m, 'del_auc'], 4), round(v5m1.loc[m5, 'deletion_AUC_excess'], 4))
           for m, m5 in [('SHAP-200', 'SHAP'), ('LIME', 'LIME'), ('IG', 'IG'), ('PFI', 'PFI'), ('PDP', 'PDP')]})
E3.to_csv(f'{OUT}/E3_insample_faithfulness.csv', index=False)
record('E3_insample_faithfulness', E3.to_dict('records'))

In [ ]:
# ===== E4. Out-of-fold faithfulness and fold stability =====
# The full-cohort model has memorised its training patients (in-sample C = 0.998), so E3's numbers describe a model
# scoring patients it has seen. Here each of the five v5 fold models explains ITS OWN HELD-OUT patients, and every
# reference quantity (KernelSHAP / GradientSHAP background, IG / DeepLIFT baseline, LIME training data, deletion fill-in)
# comes from that fold's TRAINING patients. PFI and PDP rankings are computed on the training patients (PFI uses labels).
def fold_arrays(f):
    tr, te = FOLDS[f]
    sc, sg = FOLD_SCALERS[f]
    X_tr = np.hstack([sc.transform(Xc_raw[tr]), sg.transform(Xg_clip[tr])]).astype(f32)
    X_te = np.hstack([sc.transform(Xc_raw[te]), sg.transform(Xg_clip[te])]).astype(f32)
    return tr, te, X_tr, X_te


FOLD_EXPL = []
for f in range(5):
    tr, te, X_tr, X_te = fold_arrays(f)
    Ff = Flat(FOLD_MODELS[f]).eval(); fn = np_fn(Ff)
    med_tr = np.median(X_tr, 0).astype(f32)
    bg_tr = shap.sample(X_tr, 30)
    ck = load(f'E4_fold{f}')
    if ck is None:
        ck = {'local': {}, 'runtime': {}}
        save(f'E4_fold{f}', ck)
    jobs = {'SHAP-200': lambda: kernel_shap(fn, X_te, X_tr, SHAP_LO, seed=42),
            'SHAP-4096': lambda: kernel_shap(fn, X_te, X_tr, SHAP_HI, seed=42, l1=False),
            'LIME': lambda: lime_local(fn, X_te, X_tr, num_samples=300, seed=42),
            'IG': lambda: ig_attr(Ff, X_te, med_tr),
            'GradSHAP': lambda: gradshap_attr(Ff, X_te, bg_tr, seed=42),
            'DeepLIFT': lambda: deeplift_attr(Ff, X_te, med_tr)[0]}
    for m, job in jobs.items():
        if m not in ck['local']:
            t0 = time.time(); ck['local'][m] = job(); ck['runtime'][m] = time.time() - t0
            save(f'E4_fold{f}', ck)
    if 'PFI' not in ck:
        ck['PFI'] = pfi_global(fn, X_tr, T[tr], E_int[tr], n_repeats=max(2, PFI_REPEATS // 2))
        save(f'E4_fold{f}', ck)
    if 'PDP' not in ck:
        g_tr = {m: np.abs(ck['local'][m]).mean(0) for m in ['SHAP-200', 'LIME', 'IG']}
        cand = sorted(set().union(*[set(np.argsort(-g)[:25]) for g in g_tr.values()], set(np.argsort(-ck['PFI'])[:25])))
        ck['PDP'] = pdp_global(fn, X_tr, cand); ck['PDP_cand'] = cand
        save(f'E4_fold{f}', ck)
    ck['GI'] = {**{m: np.abs(ck['local'][m]).mean(0) for m in LOCAL_M}, 'PFI': ck['PFI'], 'PDP': ck['PDP']}
    FOLD_EXPL.append(ck)
    print(f"  fold {f + 1}/5 explained ({len(te)} held-out patients)")

# Out-of-fold deletion / insertion (fill = training-fold median or mean), CIs from resampling within folds
E4_tabs = []
for mode in ['median', 'mean']:
    ck = load(f'E4_delins_{mode}')
    if ck is None:
        parts = []
        for f in range(5):
            tr, te, X_tr, X_te = fold_arrays(f)
            fn = np_fn(Flat(FOLD_MODELS[f]).eval())
            orders = {m: np.argsort(-FOLD_EXPL[f]['GI'][m]) for m in METHODS}
            parts.append((T[te], E_int[te], make_stack(fn, X_te, orders, mode, X_tr, seed=42 + f)))
        point, boot = evaluate_stacks(parts, len(METHODS))
        ck = {'point': point, 'boot': boot}
        save(f'E4_delins_{mode}', ck)
    E4_tabs.append(faith_table(ck['point'], ck['boot'], f'OOF-{mode}'))
    print(f"  held-out C of the unmodified fold models (mean over folds): {ck['point']['c0']:.4f}")

# Out-of-fold comprehensiveness / sufficiency (each patient explained once, by the model that did not see them)
comp_all, suff_all = [], []
for f in range(5):
    tr, te, X_tr, X_te = fold_arrays(f)
    fn = np_fn(Flat(FOLD_MODELS[f]).eval())
    attrs = {m: (FOLD_EXPL[f]['local'][m] if m in LOCAL_M else FOLD_EXPL[f]['GI'][m]) for m in METHODS}
    cp, sf = comp_suff(fn, X_te, attrs, 'median', X_tr, fn(X_tr).std())
    comp_all.append(cp); suff_all.append(sf)
comp_oof, suff_oof = pd.concat(comp_all, ignore_index=True), pd.concat(suff_all, ignore_index=True)
p_c, p_s = vs_ig_wilcoxon(comp_oof, True), vs_ig_wilcoxon(suff_oof, False)
E4cs = pd.DataFrame({'method': METHODS, 'comp_oof': comp_oof.mean().values, 'comp_oof_p_vs_IG': [p_c.get(m, np.nan) for m in METHODS],
                     'suff_oof': suff_oof.mean().values, 'suff_oof_p_vs_IG': [p_s.get(m, np.nan) for m in METHODS]})

# Fold stability (same protocol for all 8 methods): pairwise Spearman of full rankings and top-10 Jaccard
stab = []
for m in METHODS:
    G = [FOLD_EXPL[f]['GI'][m] for f in range(5)]
    if m == 'PDP':
        rhos = [np.nan]
    else:
        rhos = [spearmanr(G[a], G[b]).correlation for a, b in combinations(range(5), 2)]
    jac = [jaccard(np.argsort(-G[a])[:10], np.argsort(-G[b])[:10]) for a, b in combinations(range(5), 2)]
    stab.append({'method': m, 'fold_rank_rho': np.nanmean(rhos), 'fold_rank_rho_sd': np.nanstd(rhos),
                 'fold_jaccard10': np.mean(jac), 'fold_jaccard10_sd': np.std(jac)})
E4stab = pd.DataFrame(stab)
E4 = pd.concat(E4_tabs, ignore_index=True)
print("\nOut-of-fold faithfulness (fold models on their own held-out patients):")
print(E4.round(4).to_string(index=False))
print(f"\nOut-of-fold comprehensiveness / sufficiency (n = {len(comp_oof)}, Holm-corrected Wilcoxon vs IG):")
print(E4cs.round(4).to_string(index=False))
print("\nFold stability (training-fold references for every method):")
print(E4stab.round(3).to_string(index=False))
E4.to_csv(f'{OUT}/E4_oof_faithfulness.csv', index=False)
E4cs.to_csv(f'{OUT}/E4_oof_comp_suff.csv', index=False)
E4stab.to_csv(f'{OUT}/E4_fold_stability.csv', index=False)
record('E4_oof', {'faith': E4.to_dict('records'), 'comp_suff': E4cs.to_dict('records'), 'stability': E4stab.to_dict('records')})

In [ ]:
# ===== E5. ROAR-style retraining (Hooker et al. 2019) =====
# Remove (set to a constant) each method's top-k features from the full-cohort ranking, RETRAIN the gated fusion
# network with the v5 5-fold protocol, and record the out-of-fold C-index. Unlike deletion, the model never sees
# out-of-distribution inputs. Compared with removing k random features. Caveat: rankings come from the full-cohort
# model, so the removed set was chosen with all patients in view (stated in the paper).
def run_cv_gated(Xc, Xg, seed=42):
    cs = []
    for tr, te in FOLDS:
        sc, sg = RobustScaler(), RobustScaler()
        Xc_tr, Xc_te = sc.fit_transform(Xc[tr]).astype(f32), sc.transform(Xc[te]).astype(f32)
        Xg_tr, Xg_te = sg.fit_transform(Xg[tr]).astype(f32), sg.transform(Xg[te]).astype(f32)
        torch.manual_seed(seed)
        m = train_model(GatedFusion(Xc.shape[1], Xg.shape[1]), Xc_tr, Xg_tr, T[tr], E[tr])
        cs.append(harrell_c(T[te], E_int[te], risk_of(m, Xc_te, Xg_te)))
    return cs


def roar(drop_idx):
    Xc, Xg = Xc_raw.copy(), Xg_clip.copy()
    for j in drop_idx:
        if j < NC:
            Xc[:, j] = np.median(Xc[:, j])
        else:
            Xg[:, j - NC] = np.median(Xg[:, j - NC])
    return run_cv_gated(Xc, Xg)


R = load('E5_roar') or {}
if 'base' not in R:
    R['base'] = roar([]); save('E5_roar', R)
print(f"ROAR baseline (nothing removed) OOF C = {np.mean(R['base']):.4f} (v5 gated fusion: 0.5958)")
rs = np.random.RandomState(11)
RAND_SETS = {k: [rs.choice(len(FEATURES), k, replace=False) for _ in range(ROAR_RANDOM)] for k in ROAR_K}
for k in ROAR_K:
    for m in METHODS:
        if (m, k) not in R:
            R[(m, k)] = roar(np.argsort(-GI[m])[:k]); save('E5_roar', R)
    for r in range(ROAR_RANDOM):
        if ('random', k, r) not in R:
            R[('random', k, r)] = roar(RAND_SETS[k][r]); save('E5_roar', R)
    print(f"  k = {k} done")
rows = []
base = np.array(R['base'])
for k in ROAR_K:
    rnd = np.mean([np.mean(base - np.array(R[('random', k, r)])) for r in range(ROAR_RANDOM)])
    for m in METHODS:
        d = base - np.array(R[(m, k)])
        rows.append({'method': m, 'k': k, 'oof_C_after_removal': np.mean(R[(m, k)]), 'drop': d.mean(),
                     'drop_fold_sd': d.std(), 'random_drop': rnd, 'excess_drop_vs_random': d.mean() - rnd})
E5 = pd.DataFrame(rows)
print(E5.round(4).to_string(index=False))
print("Fold-level SD of the OOF C-index is ~0.06, so differences between methods of < 0.02 are within noise.")
E5.to_csv(f'{OUT}/E5_roar.csv', index=False)
record('E5_roar', E5.to_dict('records'))

In [ ]:
# ===== E6. Sanity check, robustness, linear ground truth and complexity for all 8 methods =====
# Existing methods reuse the saved v5 computations; the three new methods are computed with identical settings.

# --- (a) Model-randomisation sanity check: same randomised network as v5 (seed 0)
model_rand = randomise(model_full, seed=0)
F_rand = Flat(model_rand).eval(); fn_rand = np_fn(F_rand)
c_rand = harrell_c(T, E_int, fn_rand(X_all))
print(f"Randomised model C = {c_rand:.3f} (v5: {SAN5['c_rand']:.3f})")
RAND = {'SHAP-200': SAN5['rand_glob']['SHAP'], 'LIME': SAN5['rand_glob']['LIME'], 'IG': SAN5['rand_glob']['IG'],
        'PFI': SAN5['rand_glob']['PFI'], 'PDP': SAN5['rand_glob']['PDP']}
ck = load('E6_sanity')
if ck is None:
    ck = {'SHAP-4096': np.abs(kernel_shap(fn_rand, X_all, X_all, SHAP_HI, seed=42, l1=False)).mean(0),
          'GradSHAP': np.abs(gradshap_attr(F_rand, X_all, BG30, seed=42)).mean(0),
          'DeepLIFT': np.abs(deeplift_attr(F_rand, X_all, MED_ALL)[0]).mean(0)}
    save('E6_sanity', ck)
RAND.update(ck)
san = []
for m in METHODS:
    mask = np.isin(np.arange(len(FEATURES)), CAND) if m == 'PDP' else np.ones(len(FEATURES), bool)
    san.append({'method': m, 'sanity_abs_rho': abs(spearmanr(GI[m][mask], RAND[m][mask]).correlation),
                'sanity_top10_genes_shared': len(set(np.argsort(-GI[m])[:10]) & set(np.argsort(-RAND[m])[:10])),
                'chance_genes_shared': round(100 / (len(CAND) if m == 'PDP' else len(FEATURES)), 2)})
SAN = pd.DataFrame(san)

# --- (b) Max-sensitivity: the same 30 patients and the same perturbations as v5 (regenerated from seed 42)
N_ROB, N_PERT, SIGMA = 30, 5, 0.05
rsr = np.random.RandomState(42)
sub = rsr.choice(N, N_ROB, replace=False)
assert np.array_equal(sub, ROB5['sub']), "Robustness patients differ from v5"
iqr = np.subtract(*np.percentile(X_all, [75, 25], axis=0))
X0 = np.repeat(X_all[sub], N_PERT, axis=0)
Xp = (X0 + rsr.normal(0, 1, X0.shape) * SIGMA * iqr).astype(f32)
ROB = {'pert': {'SHAP-200': ROB5['pert']['SHAP'], 'LIME': ROB5['pert']['LIME'], 'IG': ROB5['pert']['IG']},
       'rerun': {'SHAP-200': ROB5['rerun']['SHAP'], 'LIME': ROB5['rerun']['LIME'], 'IG': ROB5['rerun']['IG']}}
for tag, XX in [('pert', Xp), ('rerun', X0)]:
    ck = load(f'E6_rob_{tag}') or {}
    jobs = {'SHAP-4096': lambda: kernel_shap(pred_full, XX, X_all, SHAP_HI, seed=7, l1=False),
            'GradSHAP': lambda: gradshap_attr(F_full, XX, BG30, seed=7),
            'DeepLIFT': lambda: deeplift_attr(F_full, XX, MED_ALL)[0]}
    for m, job in jobs.items():
        if m not in ck:
            ck[m] = job(); save(f'E6_rob_{tag}', ck)
    ROB[tag].update(ck)
sens, floor = {}, {}
for m in LOCAL_M:
    a0 = LOCAL[m][sub]
    for tag, store in [('pert', sens), ('rerun', floor)]:
        ap = np.asarray(ROB[tag][m]).reshape(N_ROB, N_PERT, -1)
        store[m] = np.linalg.norm(ap - a0[:, None], axis=2).max(1) / (np.linalg.norm(a0, axis=1) + 1e-12)
sens_df, floor_df = pd.DataFrame(sens), pd.DataFrame(floor)
p_rob = vs_ig_wilcoxon(sens_df, False)
ROBT = pd.DataFrame({'method': LOCAL_M, 'max_sens_median': sens_df.median().values,
                     'rerun_floor_median': floor_df.median().values,
                     'max_sens_p_vs_IG': [p_rob.get(m, np.nan) for m in LOCAL_M]})

# --- (c) Analytic ground truth: the v5 penalised linear Cox model (exact attributions known)
beta = LIN5['beta']
lin = LinearFlat(beta).eval(); fn_lin = np_fn(lin)
LG = {'SHAP-200': LIN5['glob']['SHAP'], 'LIME': LIN5['glob']['LIME'], 'IG': LIN5['glob']['IG'],
      'PFI': LIN5['glob']['PFI'], 'PDP': LIN5['glob']['PDP']}
ck = load('E6_linear')
if ck is None:
    ck = {'SHAP-4096': np.abs(kernel_shap(fn_lin, X_all, X_all, SHAP_HI, seed=42, l1=False)).mean(0),
          'GradSHAP': np.abs(gradshap_attr(lin, X_all, BG30, seed=42)).mean(0),
          'DeepLIFT': np.abs(deeplift_attr(lin, X_all, MED_ALL)[0]).mean(0)}
    save('E6_linear', ck)
LG.update(ck)
truth = np.abs(beta) * np.abs(X_all - MED_ALL).mean(0)
support = (beta != 0).astype(int)
top_true = set(np.argsort(-truth)[:10])
lgt = []
for m in METHODS:
    g = LG[m]
    mask = np.isin(np.arange(len(FEATURES)), LIN5['cand']) if m == 'PDP' else np.ones(len(FEATURES), bool)
    sup = (support == 1) & mask
    lgt.append({'method': m, 'linear_rank_rho_true_genes': spearmanr(g[sup], truth[sup]).correlation,
                'linear_precision@10': len(set(np.argsort(-g)[:10]) & top_true) / 10,
                'linear_AUPRC': average_precision_score(support, g)})
LGT = pd.DataFrame(lgt)


# --- (d) Complexity
def complexity(g, support_idx=None):
    a = np.abs(g if support_idx is None else g[support_idx]); q = a / a.sum(); qn = q[q > 0]
    H = -(qn * np.log(qn)).sum()
    return H / np.log(len(a)), int(np.searchsorted(np.cumsum(np.sort(q)[::-1]), 0.8) + 1)


CPX = pd.DataFrame([{'method': m, **dict(zip(['entropy', 'n_features_80pct'],
                                             complexity(GI[m], CAND if m == 'PDP' else None)))} for m in METHODS])
E6 = SAN.merge(ROBT, on='method', how='left').merge(LGT, on='method').merge(CPX, on='method')
E6['runtime_s'] = [RUNTIME[m] for m in E6.method]
print(E6.round(4).to_string(index=False))
print(f"Linear model: {support.sum()} non-zero coefficients; random-ranking AUPRC = {support.mean():.3f}.")
E6.to_csv(f'{OUT}/E6_sanity_robustness_linear.csv', index=False)
record('E6', E6.to_dict('records'))

In [ ]:
# ===== E7. Planted ground truth: more replicates, two signal strengths, all 8 methods =====
# Same simulation as v5 cell E5 (real expression matrix, Weibull times from 10 planted genes: 8 linear + 1 interacting
# pair, cohort-matched median survival and censoring). Seeds 1000-1002 at strength 1.5 reproduce v5's three replicates.
# Strength 3.0 tests whether recovery improves when the network can actually learn the signal.
# SHAP-4096 is computed on a fixed subset of SYN_SHAP_HI_N patients per replicate (cost); all other methods on all 177.
N_CAUSAL = 10
G_corr = np.nan_to_num(np.corrcoef(X_all[:, NC:].T))
SYN_IDX = np.sort(np.random.RandomState(5).choice(N, SYN_SHAP_HI_N, replace=False))


def simulate_outcome(seed, effect):
    rsi = np.random.RandomState(seed)
    G = X_all[:, NC:]
    sd = G.std(0)
    eligible = np.where((sd > 0.5) & (sd < 2.0))[0]
    if len(eligible) < 50:
        eligible = np.arange(G.shape[1])
    S = np.sort(rsi.choice(eligible, N_CAUSAL, replace=False))
    lin_, (a, b) = S[:-2], S[-2:]
    beta_ = rsi.uniform(0.5, 1.0, len(lin_)) * rsi.choice([-1, 1], len(lin_))
    eta = G[:, lin_] @ beta_ + 1.5 * np.tanh(G[:, a] * G[:, b])
    eta = effect * (eta - eta.mean()) / eta.std()
    nu = 1.5; lam = np.log(2) / np.median(T) ** nu
    Tt = (-np.log(rsi.uniform(size=N)) / (lam * np.exp(eta))) ** (1 / nu)
    target, u = 1 - E_int.mean(), rsi.uniform(size=N)
    lo, hi = 1e-3, 50 * Tt.max()
    for _ in range(60):
        cmax = (lo + hi) / 2
        lo, hi = (cmax, hi) if np.mean(u * cmax < Tt) > target else (lo, cmax)
    Cn = u * cmax
    return np.minimum(Tt, Cn).astype(f32), (Tt <= Cn).astype(f32), S + NC, eta


def fit_gated(Ts, Es, idx):
    torch.manual_seed(42)
    return train_model(GatedFusion(), X_all[idx, :NC], X_all[idx, NC:], Ts[idx], Es[idx]).eval()


def recovery(g, S):
    y = np.zeros(len(FEATURES), int); y[S] = 1
    sel = np.argsort(-g)[:10]; tp = len(set(sel) & set(S))
    out = {'AUPRC': average_precision_score(y, g), 'F1@10': tp / 10}       # k = |S| = 10, so P = R = F1
    sel_g = sel[sel >= NC] - NC; Sg = S - NC
    if len(sel_g):
        p_c = np.mean([np.abs(G_corr[j, Sg]).max() > 0.8 for j in sel_g]) * len(sel_g) / 10
        r_c = np.mean([np.abs(G_corr[s, sel_g]).max() > 0.8 for s in Sg])
        out['F1@10_corr_tolerant'] = 2 * p_c * r_c / (p_c + r_c) if p_c + r_c else 0.0
    else:
        out['F1@10_corr_tolerant'] = 0.0
    return out


syn = load('E7_planted') or []
done = {(r['effect'], r['seed']) for r in syn}
for effect, seeds in SYN_SETTINGS:
    for seed in seeds:
        if (effect, seed) in done:
            continue
        t0 = time.time()
        Ts, Es, S, eta = simulate_outcome(seed, effect)
        cv_c, or_c = [], []
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=42).split(X_all, Es):
            mm = fit_gated(Ts, Es, tr)
            cv_c.append(harrell_c(Ts[te], Es[te].astype(int), risk_of(mm, X_all[te, :NC], X_all[te, NC:])))
            or_c.append(harrell_c(Ts[te], Es[te].astype(int), eta[te]))
        msyn = fit_gated(Ts, Es, np.arange(N)); Fs = Flat(msyn).eval(); fs = np_fn(Fs)
        g = {'SHAP-200': np.abs(kernel_shap(fs, X_all, X_all, SHAP_LO, seed=42)).mean(0),
             'SHAP-4096': np.abs(kernel_shap(fs, X_all[SYN_IDX], X_all, SHAP_HI, seed=42, l1=False)).mean(0),
             'LIME': np.abs(lime_local(fs, X_all, X_all, num_samples=300)).mean(0),
             'IG': np.abs(ig_attr(Fs, X_all, MED_ALL)).mean(0),
             'GradSHAP': np.abs(gradshap_attr(Fs, X_all, BG30, seed=42)).mean(0),
             'DeepLIFT': np.abs(deeplift_attr(Fs, X_all, MED_ALL)[0]).mean(0),
             'PFI': pfi_global(fs, X_all, Ts, Es.astype(int))}
        cand = sorted(set().union(*[set(np.argsort(-g[m])[:25]) for m in ['SHAP-200', 'LIME', 'IG', 'PFI']]))
        g['PDP'] = pdp_global(fs, X_all, cand)
        for m in METHODS:
            syn.append({'effect': effect, 'seed': seed, 'method': m, **recovery(g[m], S),
                        'model_cv_C': np.mean(cv_c), 'oracle_cv_C': np.mean(or_c)})
        save('E7_planted', syn)
        print(f"  strength {effect}, seed {seed} ({time.time() - t0:.0f}s): model CV C = {np.mean(cv_c):.3f}, "
              f"oracle {np.mean(or_c):.3f}; F1@10 " + ", ".join(f"{r['method']}={r['F1@10']:.1f}" for r in syn[-len(METHODS):]))

SYN = pd.DataFrame(syn)
E7 = SYN.groupby(['effect', 'method'], sort=False)[['F1@10', 'F1@10_corr_tolerant', 'AUPRC']].agg(['mean', 'std'])
E7.columns = [f'{a}_{b}' for a, b in E7.columns]
E7 = E7.reset_index()
learn = SYN.groupby('effect')[['model_cv_C', 'oracle_cv_C']].mean()
tests = []
for effect in SYN.effect.unique():
    W = SYN[SYN.effect == effect].pivot(index='seed', columns='method', values='AUPRC')
    if len(W) >= 5:
        p = vs_ig_wilcoxon(W, True)
        tests += [{'effect': effect, 'method': m, 'AUPRC_p_vs_IG': p.get(m, np.nan)} for m in METHODS]
if tests:
    E7 = E7.merge(pd.DataFrame(tests), on=['effect', 'method'], how='left')
print(f"Random-ranking AUPRC = F1@10 = {N_CAUSAL / len(FEATURES):.3f}")
print("Learnability (mean over replicates):"); print(learn.round(3).to_string())
print(E7.round(3).to_string(index=False))
v5_reps = SYN[(SYN.effect == 1.5) & SYN.seed.isin([1000, 1001, 1002])].groupby('seed').model_cv_C.first()
print(f"Reproduction of v5 replicates (model CV C; v5: 0.622, 0.739, 0.670): {v5_reps.round(3).tolist()}")
E7.to_csv(f'{OUT}/E7_planted_summary.csv', index=False); SYN.to_csv(f'{OUT}/E7_planted_replicates.csv', index=False)
record('E7_planted', {'summary': E7.to_dict('records'), 'learnability': learn.reset_index().to_dict('records')})

In [ ]:
# ===== E8. Biological validation: Moffitt basal-like vs classical tumour subtypes (Moffitt et al. 2015) =====
# Subtype score = mean z-score of the 25 basal-like genes minus that of the 25 classical genes (higher = more basal).
# Patients are also split into two groups by k-means on the 50 genes. Questions: is the subtype prognostic here; does
# the network's out-of-fold risk track it; does the network add information beyond it; do the top attributed genes
# overlap the subtype genes more than chance; and is the picture the same within histologically confirmed PDAC?
MOFFITT_BASAL = ['VGLL1', 'UCA1', 'S100A2', 'LY6D', 'SPRR3', 'SPRR1B', 'LEMD1', 'KRT15', 'CTSL2', 'DHRS9', 'AREG',
                 'CST6', 'SERPINB3', 'KRT6C', 'KRT6A', 'SERPINB4', 'FAM83A', 'SCEL', 'FGFBP1', 'KRT7', 'KRT17',
                 'GPR87', 'TNS4', 'SLC2A1', 'ANXA8L2']
MOFFITT_CLASSICAL = ['BTNL8', 'FAM3D', 'ATAD4', 'AGR3', 'CTSE', 'LOC400573', 'LYZ', 'TFF2', 'TFF1', 'ANXA10', 'LGALS4',
                     'PLA2G10', 'CEACAM6', 'VSIG2', 'TSPAN8', 'ST6GALNAC1', 'AGR2', 'TFF3', 'CYP3A7', 'MYO1A', 'CLRN3',
                     'KRT20', 'CDH17', 'SPINK4', 'REG4']
ALIASES = {'CTSL2': ['CTSV'], 'ANXA8L2': ['ANXA8L1', 'ANXA8'], 'ATAD4': ['PRR15L']}
STUDY = 'paad_tcga_pan_can_atlas_2018'
RNA_LOCAL = f'{DATA_DIR}/{STUDY}__data_mrna_seq_v2_rsem.txt'


def full_log2_expression():
    """All genes (no expression filter), log2(RSEM + 1), rows in the paper's patient order."""
    ck = load('E8_log2_all')
    if ck is not None:
        return ck
    if not os.path.exists(RNA_LOCAL):
        url = f'https://media.githubusercontent.com/media/cBioPortal/datahub/master/public/{STUDY}/data_mrna_seq_v2_rsem.txt'
        print("Downloading the RNA file from the cBioPortal datahub (one-off)...")
        urllib.request.urlretrieve(url, RNA_LOCAL)
    rna = pd.read_csv(RNA_LOCAL, sep='\t')
    rna = rna.dropna(subset=['Hugo_Symbol']).drop(columns=[c for c in ['Entrez_Gene_Id'] if c in rna.columns])
    ex = rna.groupby('Hugo_Symbol').mean().T
    ex = ex[ex.index.str[13:15] == '01']
    ex.index = ex.index.str[:12]
    ex = ex[~ex.index.duplicated(keep='first')].loc[PID].astype(f32)
    out = np.log2(ex.clip(lower=0) + 1)
    save('E8_log2_all', out)
    return out


LOG = full_log2_expression()


def resolve(genes):
    found = {}
    for g in genes:
        for cand in [g] + ALIASES.get(g, []):
            if cand in LOG.columns and LOG[cand].std() > 0:
                found[g] = cand; break
    return found


bas, cla = resolve(MOFFITT_BASAL), resolve(MOFFITT_CLASSICAL)
Z = (LOG - LOG.mean()) / (LOG.std() + 1e-8)
score = (Z[list(bas.values())].mean(1) - Z[list(cla.values())].mean(1)).values
km = KMeans(2, n_init=50, random_state=0).fit(Z[list(bas.values()) + list(cla.values())].values)
lab = km.labels_
basal = lab == np.argmax([score[lab == c].mean() for c in (0, 1)])
print(f"Moffitt genes found: basal-like {len(bas)}/25, classical {len(cla)}/25; "
      f"k-means: {basal.sum()} basal-like, {(~basal).sum()} classical")

oof_r = np.empty(N)
for f, (tr, te) in enumerate(FOLDS):
    oof_r[te] = OOF5['Gated fusion']['te'][f]
hist_csv = f'{PARTD_DIR}/D2_patient_histology_purity.csv'
pdac = None
if os.path.exists(hist_csv):
    h = pd.read_csv(hist_csv).set_index('PATIENT_ID').reindex(PID)
    pdac = h['histology'].isin(['ductal', 'adenocarcinoma NOS']).values
subsets = {'All 177': np.ones(N, bool)}
if pdac is not None:
    subsets['PDAC histology'] = pdac
rows = []
for name, msk in subsets.items():
    lr = logrank_test(T[msk & basal], T[msk & ~basal], E[msk & basal], E[msk & ~basal]).p_value
    d = pd.DataFrame({'T': T[msk], 'E': E[msk], 'oof_risk': (oof_r[msk] - oof_r[msk].mean()) / oof_r[msk].std(),
                      'basal_score': (score[msk] - score[msk].mean()) / score[msk].std()})
    cph = CoxPHFitter(penalizer=0.0).fit(d, 'T', 'E')
    rows.append({'cohort': name, 'n': int(msk.sum()), 'n_basal': int((msk & basal).sum()),
                 'logrank_p_basal_vs_classical': lr,
                 'C_basal_score': harrell_c(T[msk], E_int[msk], score[msk]),
                 'C_network_oof': harrell_c(T[msk], E_int[msk], oof_r[msk]),
                 'spearman_oof_risk_vs_score': spearmanr(oof_r[msk], score[msk]).correlation,
                 'HR_network_adj_for_subtype': cph.hazard_ratios_['oof_risk'], 'p_network_adj': cph.summary.loc['oof_risk', 'p'],
                 'HR_subtype_adj_for_network': cph.hazard_ratios_['basal_score'], 'p_subtype_adj': cph.summary.loc['basal_score', 'p']})
E8a = pd.DataFrame(rows)

# Attribution side: do the top-attributed genes overlap the subtype genes, and do they track the subtype score?
moff_in_model = [g for g in list(bas.values()) + list(cla.values()) if g in gene_cols]
rows = []
for m in ['IG', 'SHAP-4096', 'GradSHAP', 'DeepLIFT', 'LIME', 'PFI']:
    top50 = top_genes(GI[m], k=50)
    hit = len(set(top50) & set(moff_in_model))
    rows.append({'method': m, 'moffitt_genes_in_model': len(moff_in_model), 'moffitt_in_top50': hit,
                 'expected_by_chance': round(50 * len(moff_in_model) / len(gene_cols), 2),
                 'hypergeom_p': hypergeom.sf(hit - 1, len(gene_cols), len(moff_in_model), 50) if hit else 1.0,
                 'top50_moffitt_genes': ', '.join(g for g in top50 if g in moff_in_model)})
E8b = pd.DataFrame(rows)
ig_gen = LOCAL['IG'][:, NC:].sum(1)
E8c = pd.DataFrame([{'gene': g, 'spearman_expr_vs_basal_score': spearmanr(Xg_log[:, gene_cols.index(g)], score).correlation}
                    for g in top_genes(GI['IG'], k=10)])
print(E8a.round(4).to_string(index=False))
print(f"\nPer-patient IG genomic contribution vs basal score: rho = {spearmanr(ig_gen, score).correlation:.3f}")
print(E8b.round(4).to_string(index=False))
print("\nIG top-10 genes vs the basal-like score:"); print(E8c.round(3).to_string(index=False))
pd.DataFrame({'PATIENT_ID': PID, 'basal_score': score, 'kmeans_basal': basal}).to_csv(f'{OUT}/E8_patient_subtypes.csv', index=False)
E8a.to_csv(f'{OUT}/E8_subtype_prognosis.csv', index=False); E8b.to_csv(f'{OUT}/E8_subtype_attribution.csv', index=False)
E8c.to_csv(f'{OUT}/E8_top_genes_vs_subtype.csv', index=False)
record('E8_moffitt', {'prognosis': E8a.to_dict('records'), 'attribution': E8b.to_dict('records'),
                      'ig_genomic_vs_score_rho': spearmanr(ig_gen, score).correlation, 'top10': E8c.to_dict('records'),
                      'genes_found': {'basal': len(bas), 'classical': len(cla)}})

In [ ]:
# ===== E9. Summary: Table 5 in two parts (faithfulness; reliability), ranks, LaTeX, JSON =====
def col(df, value, key='method', where=None):
    d = df if where is None else df[where(df)]
    return d.set_index(key)[value].reindex(METHODS)


ins = lambda fill: (lambda d: d.fill == fill)
roar_k = 50 if 50 in ROAR_K else ROAR_K[-1]
A = pd.DataFrame(index=METHODS)
P = pd.DataFrame(index=METHODS)            # Holm p vs IG (NaN = no test)
A['Faith@10'] = col(E3, 'faith@10', where=ins('median'))
for fill, lab in [('median', 'Del. (median)'), ('mean', 'Del. (mean)'), ('resample', 'Del. (resample)')]:
    A[lab] = col(E3, 'del_auc', where=ins(fill)); P[lab] = col(E3, 'del_p_vs_IG', where=ins(fill))
A['Ins. (resample)'] = col(E3, 'ins_auc', where=ins('resample')); P['Ins. (resample)'] = col(E3, 'ins_p_vs_IG', where=ins('resample'))
A['Del. OOF'] = col(E4, 'del_auc', where=ins('OOF-median')); P['Del. OOF'] = col(E4, 'del_p_vs_IG', where=ins('OOF-median'))
A['Ins. OOF'] = col(E4, 'ins_auc', where=ins('OOF-median')); P['Ins. OOF'] = col(E4, 'ins_p_vs_IG', where=ins('OOF-median'))
A[f'ROAR@{roar_k}'] = col(E5, 'excess_drop_vs_random', where=lambda d: d.k == roar_k)
A['Comp.'] = col(E3, 'comp', where=ins('median')); P['Comp.'] = col(E3, 'comp_p_vs_IG', where=ins('median'))
A['Suff.'] = col(E3, 'suff', where=ins('median')); P['Suff.'] = col(E3, 'suff_p_vs_IG', where=ins('median'))
A['Comp. OOF'] = col(E4cs, 'comp_oof'); P['Comp. OOF'] = col(E4cs, 'comp_oof_p_vs_IG')
A['Suff. OOF'] = col(E4cs, 'suff_oof'); P['Suff. OOF'] = col(E4cs, 'suff_oof_p_vs_IG')
HA = {c: True for c in A.columns}; HA['Suff.'] = HA['Suff. OOF'] = False

B = pd.DataFrame(index=METHODS)
Q = pd.DataFrame(index=METHODS)
B['Sanity |rho|'] = col(E6, 'sanity_abs_rho')
B['Sanity top-10'] = col(E6, 'sanity_top10_genes_shared')
for eff in sorted(E7.effect.unique()):
    B[f'Planted AUPRC ({eff:g})'] = col(E7, 'AUPRC_mean', where=lambda d, e=eff: d.effect == e)
    if 'AUPRC_p_vs_IG' in E7:
        Q[f'Planted AUPRC ({eff:g})'] = col(E7, 'AUPRC_p_vs_IG', where=lambda d, e=eff: d.effect == e)
B['Linear AUPRC'] = col(E6, 'linear_AUPRC')
B['Max-sens.'] = col(E6, 'max_sens_median'); Q['Max-sens.'] = col(E6, 'max_sens_p_vs_IG')
B['Re-run floor'] = col(E6, 'rerun_floor_median')
B['Fold rank rho'] = col(E4stab, 'fold_rank_rho')
B['Fold Jacc@10'] = col(E4stab, 'fold_jaccard10')
B['Entropy'] = col(E6, 'entropy')
B['Time (s)'] = col(E6, 'runtime_s')
HB = {c: True for c in B.columns}
for c in ['Sanity |rho|', 'Sanity top-10', 'Max-sens.', 'Re-run floor', 'Entropy', 'Time (s)']:
    HB[c] = False


def ranks(D, H):
    R = pd.DataFrame(index=D.index)
    for c in D.columns:
        v = D[c].astype(float); ok = v.notna()
        R.loc[ok, c] = rankdata(-v[ok].round(3) if H[c] else v[ok].round(3), method='min')
    R['mean rank'] = R.mean(1); R['#best'] = (R.drop(columns='mean rank') == 1).sum(1)
    return R.sort_values('mean rank')


TEX = {'Sanity |rho|': 'Sanity $|\\rho|$', 'Fold rank rho': 'Fold $\\rho$', 'Time (s)': 'Time (s)'}


def latex(D, Pv, H, caption, label, foot):
    cols = list(D.columns)
    hdr = ' & '.join('\\textbf{' + TEX.get(c, c) + '}' + ('$\\uparrow$' if H[c] else '$\\downarrow$') for c in cols)
    L = ['\\begin{table*}[t]', f'\\caption{{{caption}}}', f'\\label{{{label}}}', '\\centering\\scriptsize',
         '\\setlength{\\tabcolsep}{2pt}', '\\resizebox{\\textwidth}{!}{%', '\\begin{tabular}{l' + 'c' * len(cols) + '}',
         '\\toprule', f'\\textbf{{Method}} & {hdr} \\\\', '\\midrule']
    best = {c: D[c].astype(float).round(3).max() if H[c] else D[c].astype(float).round(3).min() for c in cols}
    for m in D.index:
        cells = []
        for c in cols:
            v = D.loc[m, c]
            if pd.isna(v):
                cells.append('---'); continue
            s = f'{v:.1f}' if c == 'Time (s)' else (f'{int(v)}' if c == 'Sanity top-10' else f'{v:.3f}')
            if round(float(v), 3) == best[c]:
                s = f'\\textbf{{{s}}}'
            p = Pv.loc[m, c] if c in Pv.columns else np.nan
            if pd.notna(p) and p < 0.05:
                s += '$^{*}$'
            cells.append(s)
        L.append(f'{m} & ' + ' & '.join(cells) + ' \\\\')
    L += ['\\bottomrule', '\\end{tabular}}', f'\\\\[2pt]\\parbox{{\\textwidth}}{{\\scriptsize {foot}}}', '\\end{table*}']
    return '\n'.join(L)


RA, RB = ranks(A, HA), ranks(B, HB)
pd.set_option('display.width', 300)
print("=== Table 5a: faithfulness ===");  print(A.round(4).to_string())
print("\nHolm p vs IG:");                  print(P.round(4).to_string())
print("\n=== Table 5b: reliability ===");  print(B.round(4).to_string())
print("\n=== Ranks 5a ==="); print(RA.to_string())
print("\n=== Ranks 5b ==="); print(RB.to_string())
tex_a = latex(A, P, HA, 'Faithfulness of eight explanation methods on the gated fusion network (TCGA-PAAD, $n=177$). '
              'Best per column in bold; $^{*}$ = differs from IG (Holm-adjusted $p<0.05$).', 'tab:xai_faith',
              'Faith@10: C-index drop after removing the top-10 features (in-sample). Del./Ins.: deletion/insertion AUC in '
              'excess of random feature sets, with removed features set to the cohort median, mean, or values resampled '
              'from other patients. OOF: each fold model explains and is evaluated on its own held-out patients. '
              f'ROAR@{roar_k}: out-of-fold C-index drop after retraining without the top-{roar_k} features, in excess of '
              'random removal. Comp./Suff.: per-patient comprehensiveness/sufficiency. SHAP-200: KernelSHAP with 200 '
              'coalitions and 50-feature L1 truncation (the configuration of our initial analysis); SHAP-4096: 4,096 '
              'coalitions, no truncation. PDP is evaluated on 57 candidate features.')
tex_b = latex(B, Q, HB, 'Reliability of eight explanation methods. Best per column in bold; $^{*}$ = differs from IG '
              '(Holm-adjusted $p<0.05$).', 'tab:xai_rel',
              'Sanity: Spearman $|\\rho|$ and number of top-10 features shared between the trained and a randomised '
              'network (chance 0.1). Planted AUPRC: recovery of 10 planted genes, mean over replicates (signal strength '
              'in brackets; random ranking = 0.01). Linear AUPRC: separation of the non-zero coefficients of a penalised '
              'linear Cox model. Max-sens.: median over 30 patients; re-run floor: the same quantity without input noise. '
              'Fold $\\rho$ / Jacc@10: agreement across the five fold models. Time: full cohort, shared CPU.')
with open(f'{OUT}/table5a_faithfulness.tex', 'w') as fh:
    fh.write(tex_a)
with open(f'{OUT}/table5b_reliability.tex', 'w') as fh:
    fh.write(tex_b)
A.to_csv(f'{OUT}/table5a_faithfulness.csv'); P.to_csv(f'{OUT}/table5a_pvalues.csv')
B.to_csv(f'{OUT}/table5b_reliability.csv'); Q.to_csv(f'{OUT}/table5b_pvalues.csv')
RA.to_csv(f'{OUT}/table5a_ranks.csv'); RB.to_csv(f'{OUT}/table5b_ranks.csv')
record('table5', {'faithfulness': A.reset_index().to_dict('records'), 'reliability': B.reset_index().to_dict('records'),
                  'ranks_a': RA.reset_index().to_dict('records'), 'ranks_b': RB.reset_index().to_dict('records')})
with open(f'{OUT}/partE_results.json', 'w') as fh:
    json.dump(RESULTS, fh, indent=2, default=lambda o: float(o) if np.isscalar(o) else str(o))
print(f"\nSaved to {OUT}. Send back the whole partE folder (zip it from Drive), or at least: partE_results.json, "
      "all E*.csv and table5*.csv/.tex files.")